In [1]:
import numpy as np
import matplotlib.pyplot as plt

from gwbench import Network, injections_CBC_params_redshift
from pprint import pprint as pp

# Import injection-parameter helpers from the waveform module.
# Adjust path if pm_waveform_np.py lives elsewhere.
import sys
sys.path.insert(0, '.')   # assumes pm_waveform_np.py is in the same directory
import pm_waveform_np as pm

/Users/ved/miniforge3/envs/neutron/lib/python3.12/site-packages/gwbench/basic_relations.py:20: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  from lal import GreenwichMeanSiderealTime


In [9]:
cosmo_dict = {'zmin':0, 'zmax':0.2, 'sampler':'uniform_comoving_volume_inversion'}

# don't really use any of this, just have to input something to injections_CBC_params_redshift
mass_dict  = {'dist':'uniform', 'mmin':0, 'mmax':0}
spin_dict  = {'dim':1, 'geom':'cartesian', 'chi_lo':0, 'chi_hi':0}

num_injs   = 1 # N, number of events
inj_id = 0
redshifted = 1
seed       = 29378
file_path  = None

injections_data = injections_CBC_params_redshift(cosmo_dict,mass_dict,spin_dict,redshifted,num_injs,seed,file_path)

rng   = np.random.default_rng(seed=seed)
M_tot = rng.uniform(2.4, 3.1)
print(f'M_tot = {M_tot:.4f} M_sun')

# Retrieve all model parameters from M_tot empirical relations
p = pm.get_params(M_tot)


# Fixed detector / extrinsic parameters
# DL   = 100.0   # luminosity distance [Mpc]
# tc   = 0.0     # coalescence time offset [s]
# phic = 0.0     # coalescence phase [rad]
# iota = np.pi / 4.0   # inclination [rad]  — 45 deg

# Assemble the full inj_params dict that GWBench will receive.
# Keys must match the argument names of hfpc (after 'f').
inj_params = {
    'fpeak_ts_'  : p['fpeak_ts'],
    'zeta_drift_': p['zeta_drift'],
    't_star_'    : p['t_star'],
    'f_spiral_'  : p['f_spiral'],
    'f_2m0_'     : p['f_2m0'],
    'f_2p0_'     : p['f_2p0'],
    'A_peak_'    : p['A_peak'],
    'A_spiral_'  : p['A_spiral'],
    'A_2m0_'     : p['A_2m0'],
    'A_2p0_'     : p['A_2p0'],
    'tau_peak_'  : p['tau_peak'],
    'tau_spiral_': p['tau_spiral'],
    'tau_2m0_'   : p['tau_2m0'],
    'tau_2p0_'   : p['tau_2p0'],
    'phi_peak_'  : p['phi_peak'],
    'phi_spiral_': p['phi_spiral'],
    'phi_2m0_'   : p['phi_2m0'],
    'phi_2p0_'   : p['phi_2p0'],
    'N_'         : p['N'],
    's_'         : p['s'],
    'DL'    : injections_data[8][inj_id],
    'tc'    : 0.,
    'phic'  : 0.,
    'iota'  : injections_data[9][inj_id],
    'ra'    : injections_data[10][inj_id],
    'dec'   : injections_data[11][inj_id],
    'psi'   : injections_data[12][inj_id],
    'z'     : injections_data[13][inj_id],
    'Mc'    : None,
    'eta'   : None
}
print('inj_params assembled.')
pp(inj_params)


M_tot = 2.5901 M_sun
inj_params assembled.
{'A_2m0_': 0.5708356417043436,
 'A_2p0_': 0.0550299485652177,
 'A_peak_': 0.5981190869549815,
 'A_spiral_': 0.3495663137703424,
 'DL': np.float64(940.514385177975),
 'Mc': None,
 'N_': 0.7688178463415951,
 'dec': np.float64(1.1829476432948405),
 'eta': None,
 'f_2m0_': 1.7062377675454572,
 'f_2p0_': 3.706237767545457,
 'f_spiral_': 2.0164068475622106,
 'fpeak_ts_': 2.5561323719978426,
 'iota': np.float64(0.2367522187410996),
 'phi_2m0_': 4.111419197897909,
 'phi_2p0_': 0.44382498220333844,
 'phi_peak_': 2.7788867771183163,
 'phi_spiral_': 3.33436548724692,
 'phic': 0.0,
 'psi': np.float64(3.277322349164817),
 'ra': np.float64(4.043179805492967),
 's_': 0.075,
 't_star_': 6.149210457332863,
 'tau_2m0_': 0.42219184814368127,
 'tau_2p0_': 2.1284230529338117,
 'tau_peak_': 8.3699903339256,
 'tau_spiral_': 1.2514438809637083,
 'tc': 0.0,
 'z': np.float64(0.18720514324244297),
 'zeta_drift_': -0.04882103046859143}


/Users/ved/miniforge3/envs/neutron/lib/python3.12/site-packages/gwbench/injections.py:471: RuntimeWarning: invalid value encountered in divide
  eta_vec = brs.eta_of_q(m1_vec/m2_vec)


In [3]:
f_lo = 1.              # Hz
f_hi = 8000.0          # Hz  (safely above ~3.5 kHz model content)
df   = 2.0 ** -4       # Hz  (~0.0625 Hz resolution — matches event_level.py convention)
f    = np.arange(f_lo, f_hi + df, df)
print(f'Frequency grid: {f_lo:.0f} – {f_hi:.0f} Hz,  {len(f)} points,  df = {df:.4f} Hz')

Frequency grid: 1 – 8000 Hz,  127985 points,  df = 0.0625 Hz


In [7]:
# Waveform model specification.
# 'np' key points to the numpy waveform file consumed by numeric derivatives.
wf_model_name    = 'pm_waveform'
wf_other_var_dic = None
# user_waveform    = {'np': 'pm_waveform_np.py'}
user_waveform    = 'pm_waveform_np.py'   # path to the numpy waveform file


# All model parameters are Fisher variables.
# Trailing underscore names match hfpc argument names exactly.
deriv_symbs_string = (
    'fpeak_ts_ zeta_drift_ t_star_ '
    'f_spiral_ f_2m0_ f_2p0_ '
    'A_peak_ A_spiral_ A_2m0_ A_2p0_ '
    'tau_peak_ tau_spiral_ tau_2m0_ tau_2p0_ '
    'phi_peak_ phi_spiral_ phi_2m0_ phi_2p0_ '
    'N_ '
    'DL tc phic iota'
)

# Analytic derivatives for the three phase/distance parameters
ana_deriv_symbs_string = 'DL tc phic'

# No cos/log reparametrisations for now
conv_cos = ('dec','iota')
conv_log = ('DL')
use_rot = 0
only_net = 1
num_cores = None
gen_derivs = None

# Numeric derivative settings
step   = 1e-6
method = 'central'
order  = 2

print('GWBench configuration set.')

GWBench configuration set.


In [10]:
# network_spec = ['CE-40_C','CE-40_S']
network_spec = 'E' # ET

net = Network(network_spec, logger_name='PM_FIM', logger_level='INFO')

net.set_net_vars(
    wf_model_name    = wf_model_name,
    wf_other_var_dic = wf_other_var_dic,
    user_waveform    = user_waveform,
    f                = f,
    inj_params       = inj_params,
    deriv_symbs_string     = deriv_symbs_string,
    ana_deriv_symbs_string = ana_deriv_symbs_string,
    conv_cos = conv_cos,
    conv_log = conv_log,
    use_rot  = use_rot,
)

net.calc_errors(
    only_net   = 1,
    derivs     = 'num',
    step       = step,
    method     = method,
    order      = order,
    gen_derivs = gen_derivs,
    num_cores  = num_cores,
)

print('Fisher run complete.')

2026-05-19 13:17:14,650 - PM_FIM - INFO : PSDs, antenna patterns, and LPFs loaded.
2026-05-19 13:17:14,650 - PM_FIM - INFO : Calculate numeric derivatives of detector responses.
2026-05-19 13:17:14,651 - PM_FIM - INFO :    ET_ET1
2026-05-19 13:17:21,439 - PM_FIM - INFO :    ET_ET2
2026-05-19 13:17:28,097 - PM_FIM - INFO :    ET_ET3
2026-05-19 13:17:34,736 - PM_FIM - INFO : Numeric derivatives of detector responses calculated.
2026-05-19 13:17:34,742 - PM_FIM - INFO : SNRs calculated.
2026-05-19 13:17:34,742 - PM_FIM - INFO : Calculate errors (Fisher & cov matrices).
2026-05-19 13:17:34,742 - PM_FIM - INFO :    ET_ET1
2026-05-19 13:17:34,929 - PM_FIM - INFO :    ET_ET2
2026-05-19 13:17:35,126 - PM_FIM - INFO :    ET_ET3
2026-05-19 13:17:35,411 - PM_FIM - WARNING : calc_errors: tag = network - 90%-credible sky area not calculated due to missing RA or DEC (COS_DEC) errors.
2026-05-19 13:17:35,411 - PM_FIM - INFO : Errors calculated.


Fisher run complete.


## 4. Results

In [19]:
# Key result: fpeak(t*) recovery
params  = list(net.deriv_variables)
sigmas  = np.sqrt(np.diag(net.cov))

idx_fp   = params.index('fpeak_ts_')
true_fp  = inj_params['fpeak_ts_']   # kHz
sigma_fp = sigmas[idx_fp]             # kHz

print(sigma_fp)
print(true_fp)

# print(f'\n=== f_peak(t*) recovery ===')
# print(f'  M_tot              = {M_tot:.4f} M_sun')
# print(f'  f_peak(t*) [true]  = {true_fp:.4f} kHz')
# print(f'  sigma_fpeak        = {sigma_fp:.6f} kHz')
print(f'  Relative error     = {sigma_fp / true_fp:.4%}')

0.09127828
2.5561323719978426
  Relative error     = 3.5710%


/var/folders/t6/9g0339bj7fjdlm1vw31kx_tc0000gn/T/ipykernel_15938/2477169882.py:3: RuntimeWarning: invalid value encountered in sqrt
  sigmas  = np.sqrt(np.diag(net.cov))


## 5. Covariance matrix heatmap

In [ ]:
cov  = net.cov
corr = cov / np.outer(sigmas, sigmas)   # correlation matrix

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap='RdBu_r')
plt.colorbar(im, ax=ax, label='Correlation')
ax.set_xticks(range(len(params)))
ax.set_yticks(range(len(params)))
ax.set_xticklabels(params, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(params, fontsize=8)
ax.set_title(f'Fisher correlation matrix  |  $M_{{\\rm tot}} = {M_tot:.3f}\,M_\\odot$,  SNR = {snr_net:.1f}')
plt.tight_layout()
plt.savefig('pm_correlation_matrix.pdf', dpi=150)
plt.show()
print('Saved pm_correlation_matrix.pdf')

## 6. 1-sigma error bar on $f_\mathrm{peak}(t^*)$

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar([M_tot], [true_fp], yerr=[sigma_fp],
            fmt='o', capsize=6, color='steelblue',
            label=rf'$f_{{\rm peak}}(t^*) = {true_fp:.3f} \pm {sigma_fp:.4f}$ kHz')
ax.set_xlabel(r'$M_{\rm tot}\;[M_\odot]$')
ax.set_ylabel(r'$f_{\rm peak}(t^*)\;[\mathrm{kHz}]$')
ax.set_title('Post-merger $f_\\mathrm{peak}(t^*)$ recovery (single event, ET)')
ax.legend()
plt.tight_layout()
plt.savefig('pm_fpeak_recovery.pdf', dpi=150)
plt.show()
print('Saved pm_fpeak_recovery.pdf')